# Task 1

## 1. IoU entre cajas

Cajas:

$$
{b} = (142, 89, 218, 165), \quad b^* = (138, 84, 222, 170)
$$

Primero sacamos la intersección (la parte donde ambas cajas se traslapan):

$$
x_{min} = \max(142,138)=142,\quad x_{max}=\min(218,222)=218
$$

$$
y_{min} = \max(89,84)=89,\quad y_{max}=\min(165,170)=165
$$

Esto nos da un cuadro común de:

$$
76 \times 76 \Rightarrow |I| = 5776
$$

Luego áreas individuales (para saber cuánto cubre cada caja):

$$
A_{pred}=5776,\quad A_{gt}=84 \times 86 = 7224
$$

La unión combina ambas áreas sin duplicar la intersección:

$$
|U| = 5776 + 7224 - 5776 = 7224
$$

Finalmente:

$$
IoU = \frac{5776}{7224} \approx 0.80
$$

### ¿Qué significa esto?

Un **IoU de 0.80** es bastante bueno: la detección está bien alineada con el producto real.  
por lo tanto le explicaría: En que no hay problema con que los numeros no parezcan matchear desde el inicio ya que tenemos un alto "encaje" del producto real con la detección por lo tanto funcionará.

---

## 2. Interpretación de la fórmula

$$
IoU = \frac{|I|}{|U|}
$$

$$
|I|
$$

: lo que ambas cajas comparten

$$
|U|
$$

: todo lo que cubren juntas

--- 

Si usáramos el ground truth:

$$
\frac{|I|}{|GT|}
$$

Con una caja enorme que cubra todos los productos del shelf seria suficiente para tener score perfecto porque si o sí estaría dentro el producto. 

Usar la unión penaliza eso automáticamente:
- cajas muy grandes → unión crece → IoU baja  
- cajas mal alineadas → menos overlap → IoU baja  


---

## 3. Elección de umbral IoU

Opciones:

$$
\theta = 0.5
$$

(relajado)
$$
\theta = 0.75
$$

(estricto)


Se prioriza el falso positivo leve que un falso negativo ya que no poder identificar un producto que si esta se empieza a perder el inventario generando problemas mas costosos que detectar una posible caja imperfecta por lo tanto el balance entre estricto y suficiente es necesario para evitar perder objetos.

---

### Recomendación final

$$
\theta = 0.5
$$

### ¿Por qué?

- Prioriza **recall** 
- Evita alertas falsas de stock vacío

En resumen:
Mejor detectar “bien suficiente” que fallar detecciones críticas.



## Pregunta 1.2
## 4. Precisión y Recall

Datos:

$$
TP = 12,\quad FP = 6,\quad FN = 3
$$

---

### Precisión

$$
P = \frac{TP}{TP + FP} = \frac{12}{12+6} = \frac{12}{18} \approx 0.67
$$

Aquí el $(TP + FP)$ son las detecciones que el modelo hizo.  
La precisión seria de cuales de las predicciones estas son correctas

---

### Recall

$$
R = \frac{TP}{TP + FN} = \frac{12}{12+3} = \frac{12}{15} = 0.80
$$

Aquí el denominador $(TP + FN)$ son  los productos reales en el anaquel  
El recall seria de lo que existía cuanto se logró detectar?

---

- Precisión ≈ **0.67** → hay ruido (algunas detecciones falsas)  
- Recall = **0.80** → detecta la mayoría de productos  

---

### ¿Por qué necesitamos ambas?

Porque miden cosas distintas:

- Precisión → calidad de las detecciones  
- Recall → cobertura del sistema  

---

## 5. Preferencia del negocio → métricas


> "Prefiero no perder quiebres de stock"

Lo cual su traducción técnica sería:

Quiere **alto recall**  
Acepta bajar precisión

---


### Ajuste práctico

Bajar el **threshold de confianza**

- Más bajo → más detecciones  
- ↑ TP pero también ↑ FP  
- ↓ FN  

---

## 6. ¿Qué es mAP?

mAP = **mean Average Precision**

No es una sola precisión, sino un promedio de desempeño considerando los dif. thresholds.

En vez de medir en un solo punto (ej: threshold fijo),  
mAP evalúa cómo se comporta el modelo en todo el rango de decisiones.

Es como ver toda la curva Precision-Recall, no solo un punto.

---

### Diferencia de protocolos

**mAP@0.5 (PASCAL VOC):**

- Solo evalúa con IoU ≥ 0.5  
- Más permisivo  
- Más fácil obtener buen score  

---

**mAP@0.5:0.95 (COCO):**

- Promedia IoU desde 0.5 hasta 0.95  
- Incluye localización muy **precisa**  
- Mucho más estricto  

---

### ¿Cuál es más exigente?

**COCO (0.5:0.95)**

Porque no solo pide detectar el objeto, sino ubicarlo muy bien.

- mAP@0.5 con suficiente para saber si hay producto  
- mAP@0.5:0.95 para evaluar qué tan bien está alineada la caja  

COCO es más exigente porque penaliza errores pequeños de localización  
(que en anaqueles pueden afectar conteo o layout)



## Pregunta 1.3
## 7. ¿Qué es NMS y por qué hay tantas cajas?

Lo que está pasando es normal en detección de objetos.

El modelo no “ve una caja por producto”, sino que prueba muchas posibles ubicaciones (anchors o candidatos) y varias coinciden sobre el mismo objeto.

NMS(Non Maximum Suppression), es un filtro que se queda solo con **la mejor caja por objeto** y elimina duplicados.

### Algoritmo

1. El modelo asigna un **score de confianza** a cada caja  
2. Se ordenan las cajas de mayor a menor score  
3. Se toma la mejor caja (la más confiable)  
4. Se eliminan todas las cajas que se traslapan mucho con ella (IoU alto)  
5. Se repite con las cajas restantes  

Resultado: una caja por producto (o pocas), en lugar de decenas.

---

## 8. Elección de θ_NMS en anaquel denso

En anaqueles:

- productos están muy juntos, normalmente tocandose  
- cajas legítimas pueden traslaparse un poco  

---

**Si θ_NMS es muy bajo (agresivo)**

- elimina muchas cajas  
- riesgo: borrar productos reales que están pegados  

mas falsos negativos  

**Si θ_NMS es alto (menos agresivo)**

- permite más overlap  
- mantiene cajas cercanas  

menos riesgo de perder productos pero mas duplicados (FP)

---

### Recomendación

Nuevamente se trata de balance, usar **θ_NMS alto (ej: 0.5 – 0.7)**

Porque:

- mejor tolerar duplicados  
- que eliminar productos reales  

En retail:

perder un producto (FN) es peor que duplicarlo (FP leve)

---

## 9. Orden: threshold vs NMS

El orden correcto es:

1. Aplicar **threshold de confianza**
2. Luego aplicar **NMS**

---

### ¿Por qué?

Primero se elimina cajas con baja confianza o sea el ruido:

- se reduce cantidad de cajas    

Luego NMS trabaja sobre un conjunto más limpio

Poruqe si se hace NMS primero:

- NMS procesa **todas las 312 cajas**  
- muchas son basura (bajo score)  
Se hace trabajo doble por esos datos basura

**En el contexto:** 

Sistema procesa:

- 30 imágenes por minuto  
- sin GPU  

Si no filtras primero:

- aumenta latencia  
- puedes pasar de <500ms a tiempos no aceptables  

